<a href="https://colab.research.google.com/github/RCalvoso/grupo4_projeto_integrador_2/blob/main/04_Visao_Computacional.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.preprocessing import image
from google.colab import drive

# 1. Garantir acesso aos arquivos
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Localizar diretório dos dados
dir_path = None
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if 'train_clean.csv' in files:
        dir_path = root
        break

train_df = pd.read_csv(os.path.join(dir_path, 'train_clean.csv'))
val_df = pd.read_csv(os.path.join(dir_path, 'val_clean.csv'))
test_df = pd.read_csv(os.path.join(dir_path, 'test_clean.csv'))

# 2. Carregar modelo pré-treinado ResNet50 sem a cabeça de classificação (include_top=False)
base_model = ResNet50(weights='imagenet', include_top=False, pooling='avg')
print("✅ ResNet50 carregada como extrator de feições visuais.")

# 3. Função para extrair embeddings de imagem
def extract_image_features(df, base_img_dir):
    features = []
    # Cria vetor de zeros caso a imagem falhe ou não exista (fallback)
    zero_vector = np.zeros(2048)

    for idx, row in df.iterrows():
        img_rel_path = row['image_path']
        if pd.isna(img_rel_path) or not img_rel_path:
            features.append(zero_vector)
            continue

        img_full_path = os.path.join(base_img_dir, str(img_rel_path))

        if os.path.exists(img_full_path):
            try:
                img = image.load_img(img_full_path, target_size=(224, 224))
                x = image.img_to_array(img)
                x = np.expand_dims(x, axis=0)
                x = preprocess_input(x)
                feat = base_model.predict(x, verbose=0).flatten()
                features.append(feat)
            except Exception:
                features.append(zero_vector)
        else:
            features.append(zero_vector)

    return np.array(features)

base_img_dir = os.path.dirname(dir_path) # pasta raiz dos dados/imagens

print("Extraindo feições das imagens de Treino...")
img_feats_train = extract_image_features(train_df, base_img_dir)

print("Extraindo feições das imagens de Validação...")
img_feats_val = extract_image_features(val_df, base_img_dir)

print("Extraindo feições das imagens de Teste...")
img_feats_test = extract_image_features(test_df, base_img_dir)

# 4. Salvar os embeddings de imagem gerados
np.save(os.path.join(dir_path, 'img_feats_train.npy'), img_feats_train)
np.save(os.path.join(dir_path, 'img_feats_val.npy'), img_feats_val)
np.save(os.path.join(dir_path, 'img_feats_test.npy'), img_feats_test)

print(f"\n✅ Embeddings de imagem extraídos com sucesso!")
print(f"Dimensão das feições de imagem: {img_feats_train.shape}")

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
✅ ResNet50 carregada como extrator de feições visuais.
Extraindo feições das imagens de Treino...
Extraindo feições das imagens de Validação...
Extraindo feições das imagens de Teste...

✅ Embeddings de imagem extraídos com sucesso!
Dimensão das feições de imagem: (3500, 2048)
